# Notebook 02: CHMM Model Fitting via Baum-Welch

This notebook trains a **continuous Gaussian Hidden Markov Model** (CHMM) on equity excess growth rates
using the Baum-Welch (Expectation-Maximization) algorithm. Unlike the discrete approach in the published paper
(arXiv:2603.10202), which bins observations into Laplace quantiles and counts transition frequencies,
Baum-Welch learns emission parameters ($\mu_k$, $\sigma_k$) and transition probabilities directly from
raw continuous returns, eliminating quantization error.

**No jump mechanisms are used here** -- this is a pure CHMM. The jump-augmented model is explored in later notebooks.

### Sections
1. Configuration (tunable hyperparameters)
2. Data loading and return computation
3. Model training via Baum-Welch
4. Convergence diagnostics
5. Emission distribution visualization
6. Transition matrix heatmap
7. Stationary distribution
8. Residence times
9. Emission parameter table
10. Viterbi decoding and regime overlay
11. Simulation and ACF comparison

## Setup

In [ ]:
include("../Include.jl");

## Configuration

Tune these parameters interactively. `K` controls model complexity (number of hidden regimes),
and `MAX_ITER` caps the EM iterations (convergence is checked via log-likelihood tolerance internally).

In [ ]:
# --- TUNABLE CONFIGURATION ---
ticker = "SPY";              # asset to model (change to any ticker in the dataset)
K = 6;                        # number of hidden states (try 4, 6, 8, 10, 13)
MAX_ITER = 60;                # maximum Baum-Welch iterations
risk_free_rate = 0.0;         # risk-free rate
Δt = 1/252;                   # daily time step in years
L = 252;                      # ACF max lag (1 trading year)

## Load Data and Compute Returns

In [ ]:
# --- LOAD IN-SAMPLE DATA (2014-2024) ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

# Filter to tickers with complete history
dataset = Dict{String,DataFrame}();
for (t, data) ∈ train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

# Compute excess log growth rates for the target ticker
R_is = log_growth_matrix(dataset, ticker; Δt=Δt, risk_free_rate=risk_free_rate);

println("Loaded $(ticker): T = $(length(R_is)) daily observations")
println("Mean = $(round(mean(R_is), digits=2)), Std = $(round(std(R_is), digits=2))")

## Train Continuous HMM via Baum-Welch

The `build` factory calls `baum_welch()` internally with quantile-based initialization,
then wraps the fitted parameters into a `MyContinuousHiddenMarkovModel` struct.

In [ ]:
# --- TRAIN MODEL ---
model = build(MyContinuousHiddenMarkovModel, (
    observations = R_is,
    number_of_states = K,
    max_iter = MAX_ITER
));

println("Training complete: $(length(model.log_likelihood_history)) EM iterations")
println("Final log-likelihood: $(round(model.log_likelihood_history[end], digits=2))")

## Convergence Plot

The log-likelihood should increase monotonically at each EM step. A plateau indicates convergence.

In [ ]:
# --- CONVERGENCE PLOT ---
p_conv = plot(model.log_likelihood_history,
    title="Baum-Welch Convergence ($(ticker), K=$(K))",
    xlabel="Iteration", ylabel="Log-Likelihood",
    legend=false, lw=2, color=:navy, marker=:circle, ms=3,
    size=(700, 400));
display(p_conv)

## Emission Distributions

Each hidden state $k$ emits observations from a Gaussian $\mathcal{N}(\mu_k, \sigma_k)$.
Overlaying these on the observed histogram shows how the mixture decomposes the marginal distribution.

In [ ]:
# --- EMISSION PDFs OVERLAY ---
x_grid = range(-800, 800, length=2000);
colors = cgrad(:RdYlBu, K, categorical=true);

p_emit = plot(title="Learned Emission Distributions (K=$K)", titlefontsize=10,
    xlabel="Excess Growth Rate", ylabel="Density", legend=:topright, size=(900, 500));

# Observed histogram
histogram!(p_emit, R_is, normalize=true, bins=150, alpha=0.3, color=:gray, label="Observed");

# Overlay each state's Gaussian PDF
for s in 1:K
    d = model.emission[s];
    plot!(p_emit, x_grid, pdf.(d, x_grid), lw=2, color=colors[s],
        label="State $s (μ=$(round(mean(d),digits=1)), σ=$(round(std(d),digits=1)))", alpha=0.85);
end

xlims!(p_emit, -800, 800);
display(p_emit)

## Transition Matrix Heatmap

The $K \times K$ transition matrix $T_{ij} = P(s_{t+1} = j \mid s_t = i)$ encodes regime persistence
(diagonal) and switching behavior (off-diagonal). Displayed in $\log_{10}$ scale to reveal
rare transitions.

In [ ]:
# --- EXTRACT TRANSITION MATRIX ---
T_mat = zeros(K, K);
for i in 1:K
    T_mat[i, :] = probs(model.transition[i]);
end

# Log10 heatmap
T_log = log10.(T_mat .+ 1e-10);

p_trans = heatmap(T_log, title="Transition Matrix (log₁₀ scale, K=$K)", titlefontsize=10,
    xlabel="To State", ylabel="From State", color=:viridis,
    yflip=true, aspect_ratio=:equal, size=(600, 500));
display(p_trans)

## Stationary Distribution

The stationary distribution $\pi$ satisfies $\pi = \pi T$. We approximate it by matrix powering: $\pi \approx e_1^\top T^{1000}$.

In [ ]:
# --- STATIONARY DISTRIBUTION ---
π_stat = (T_mat^1000)[1, :];

p_pi = bar(1:K, π_stat, title="Stationary Distribution π (K=$K)", titlefontsize=10,
    xlabel="State", ylabel="Probability", legend=false,
    color=colors[1:K], alpha=0.8, size=(700, 400));
display(p_pi)

println("Sum of π: $(round(sum(π_stat), digits=6))")

## Residence Times

The expected natural residence time in state $k$ under the Markov chain is $\tau_k = 1 / (1 - T_{kk})$.
Short residence in tail states motivates the jump mechanism in later notebooks.

In [ ]:
# --- RESIDENCE TIMES ---
residence_times = [1.0 / (1.0 - T_mat[k, k]) for k in 1:K];

p_res = bar(1:K, residence_times, title="Expected Residence Time per State (K=$K)", titlefontsize=10,
    xlabel="State", ylabel="Steps (trading days)", legend=false,
    color=colors[1:K], alpha=0.8, size=(700, 400));
display(p_res)

# Print table
println("State | T_kk      | E[τ_k] (days)")
println("------+-----------+--------------")
for k in 1:K
    println("  $(lpad(k,2))  | $(lpad(round(T_mat[k,k], digits=4),9)) | $(round(residence_times[k], digits=2))")
end

## Emission Parameters Table

Summary of the learned Gaussian emission parameters for each state, ordered by mean.

In [ ]:
# --- EMISSION PARAMETERS TABLE ---
println("State | Mean (μ)     | Std Dev (σ)  | π_stationary | Interpretation")
println("------+--------------+--------------+--------------+---------------")
for s in 1:K
    d = model.emission[s];
    μ_s = round(mean(d), digits=2);
    σ_s = round(std(d), digits=2);
    π_s = round(π_stat[s], digits=4);
    label = if s == 1 "CRASH (most negative)" elseif s == K "BOOM (most positive)" else "" end;
    println("  $(lpad(s,2))  | $(lpad(μ_s,12)) | $(lpad(σ_s,12)) | $(lpad(π_s,12)) | $label")
end

## Viterbi Decoding: Regime Overlay on Close Prices

The Viterbi algorithm finds the most likely hidden state sequence given the observations.
We overlay the decoded regimes on the in-sample close price series to visualize how the model
partitions the time series into regimes.

In [ ]:
# --- VITERBI DECODING ---
decoded_states = viterbi(R_is, model);

# Close prices (aligned with returns: skip first day)
close_prices = dataset[ticker].close[2:end];
dates = dataset[ticker].date[2:end];
T_obs = length(close_prices);

# Regime overlay plot
p_viterbi = plot(title="Viterbi Decoded Regimes on $(ticker) Close Prices", titlefontsize=10,
    xlabel="Trading Day Index", ylabel="Close Price (\$)", legend=:topleft, size=(1100, 500));

# Plot close price as thin gray line
plot!(p_viterbi, 1:T_obs, close_prices, lw=0.5, color=:gray, alpha=0.6, label="Close");

# Overlay colored scatter by regime
for s in 1:K
    idx = findall(decoded_states .== s);
    scatter!(p_viterbi, idx, close_prices[idx], ms=1.5, color=colors[s],
        label="State $s", alpha=0.7, markerstrokewidth=0);
end

display(p_viterbi)

## Simulate and Compare ACF

We simulate one path from the trained CHMM (same length as the in-sample data) and compare
the autocorrelation functions of raw returns and absolute returns between observed and simulated data.
A good model should reproduce:
- Near-zero ACF in raw returns (no linear predictability)
- Slow-decaying ACF in |returns| (volatility clustering)

In [ ]:
# --- SIMULATE ONE PATH ---
T_sim = length(R_is);
start_state = rand(1:K);  # random initial state
sim_states = model(start_state, T_sim);

# Generate returns from emissions along the simulated state path
sim_returns = [rand(model.emission[sim_states[t]]) for t in 1:T_sim];

println("Simulated $(T_sim) observations starting from state $(start_state)")
println("Sim mean = $(round(mean(sim_returns), digits=2)), Sim std = $(round(std(sim_returns), digits=2))")

In [ ]:
# --- ACF COMPARISON ---
τ = 1:(L-1);
ci = 2.576 / sqrt(T_sim);  # 99% CI

# Observed ACFs
acf_obs_raw = autocor(R_is, τ);
acf_obs_abs = autocor(abs.(R_is), τ);

# Simulated ACFs
acf_sim_raw = autocor(sim_returns, τ);
acf_sim_abs = autocor(abs.(sim_returns), τ);

# Panel 1: Raw returns ACF
p_acf1 = plot(title="(a) Raw Returns ACF", titlefontsize=10,
    xlabel="Lag (trading days)", ylabel="ACF");
plot!(p_acf1, τ, acf_obs_raw, lw=2, color=:steelblue, label="Observed");
plot!(p_acf1, τ, acf_sim_raw, lw=2, color=:red, ls=:dash, label="CHMM Simulated");
plot!(p_acf1, τ, ci .* ones(length(τ)), lw=1, color=:gray, ls=:dot, label="99% CI");
plot!(p_acf1, τ, -ci .* ones(length(τ)), lw=1, color=:gray, ls=:dot, label="");

# Panel 2: Absolute returns ACF
p_acf2 = plot(title="(b) Absolute Returns ACF", titlefontsize=10,
    xlabel="Lag (trading days)", ylabel="ACF");
plot!(p_acf2, τ, acf_obs_abs, lw=2, color=:darkorange, label="Observed |Gₜ|");
plot!(p_acf2, τ, acf_sim_abs, lw=2, color=:red, ls=:dash, label="CHMM Simulated |Gₜ|");
plot!(p_acf2, τ, ci .* ones(length(τ)), lw=1, color=:gray, ls=:dot, label="99% CI");
plot!(p_acf2, τ, -ci .* ones(length(τ)), lw=1, color=:gray, ls=:dot, label="");

# Combine
fig_acf = plot(p_acf1, p_acf2, layout=(1, 2), size=(1100, 400),
    plot_title="ACF Comparison: Observed vs CHMM (K=$K)", plot_titlefontsize=12);
display(fig_acf)

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice, investment recommendations, or trading strategies.